# 03 — Feature Engineering

**Input:** `data/processed/sfu_clean.db` — table: `offerings`  
**Output:** `data/processed/` — 9 CSVs (train/test/final × offered/capacity/enrollment)

### Pipeline
```
offerings (1 row/section)
    → aggregate → course_term (1 row/course+term, offered only)
    → expand    → full grid   (1 row/course+term, offered=0 and offered=1)
    → lookback  → historical features (no leakage: only prior terms used)
    → join      → static course features
    → encode    → categoricals + cold-start fill
    → split     → train (≤2024) / test (2025) / final (2026)
```

### Target split
| Split | Years | Purpose |
|-------|-------|----------|
| Train | 2020–2024 | Model training |
| Test | 2025 | Model selection |
| Final | 2026 Spring | Post-refit evaluation |

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

CLEAN_DB = Path('../data/processed/sfu_clean.db')
RAW_DB   = Path('../data/raw/sfu_ml.db')
OUT_DIR  = Path('../data/processed')
OUT_DIR.mkdir(exist_ok=True)

TRAIN_CUTOFF = 2024
TEST_YEAR    = 2025
FINAL_YEAR   = 2026
HIGH_FILL_THRESHOLD = 0.9  # fill_rate >= 90% counts as 'high fill'

assert CLEAN_DB.exists(), f'not found: {CLEAN_DB}'
assert RAW_DB.exists(),   f'not found: {RAW_DB}'
print('ready')

ready


---
## Block 1 — Load data

In [2]:
conn = sqlite3.connect(CLEAN_DB)
df   = pd.read_sql('SELECT * FROM offerings', conn)
conn.close()

# load terms table for chronological ordering
conn2 = sqlite3.connect(RAW_DB)
terms = pd.read_sql('SELECT * FROM ml_terms', conn2)
courses_meta = pd.read_sql(
    'SELECT ml_course_id, dept_code, course_level, degree_level, units, prereq_count FROM ml_courses',
    conn2
)
conn2.close()

# assign sequential term index (0 = earliest, 18 = latest)
# this is what we use for all 'prior terms' lookups
terms = terms.sort_values(['year', 'term_order']).reset_index(drop=True)
terms['term_idx'] = terms.index

df['fill_rate'] = df['enrolled'] / df['capacity']

print(f'offerings:    {len(df):,} rows')
print(f'terms:        {len(terms)}')
print(f'courses_meta: {len(courses_meta):,}')
print()
print('Term ordering:')
print(terms[['term_idx', 'year', 'term', 'semester_code', 'is_covid_affected']].to_string(index=False))

offerings:    33,659 rows
terms:        19
courses_meta: 3,310

Term ordering:
 term_idx  year   term  semester_code  is_covid_affected
        0  2020 spring           1201                  1
        1  2020 summer           1204                  1
        2  2020   fall           1207                  1
        3  2021 spring           1211                  1
        4  2021 summer           1214                  1
        5  2021   fall           1217                  1
        6  2022 spring           1221                  0
        7  2022 summer           1224                  0
        8  2022   fall           1227                  0
        9  2023 spring           1231                  0
       10  2023 summer           1234                  0
       11  2023   fall           1237                  0
       12  2024 spring           1241                  0
       13  2024 summer           1244                  0
       14  2024   fall           1247                  0
       15

---
## Block 2 — Aggregate sections → course-term level

We predict at the **course level** (e.g. all sections of CMPT 225 in Fall 2024 combined).  
This collapses the ~33K section rows into one row per (course, term).

In [3]:
course_term = df.groupby(['ml_course_id', 'ml_term_id']).agg(
    n_sections     = ('offering_id',  'count'),
    total_capacity = ('capacity',     'sum'),
    total_enrolled = ('enrolled',     'sum'),
    avg_fill_rate  = ('fill_rate',    'mean'),
).reset_index()

# is this term a 'high fill' term for this course?
# computed as: course-level fill rate (total enrolled / total capacity)
course_term['course_fill_rate'] = course_term['total_enrolled'] / course_term['total_capacity']
course_term['is_high_fill_term'] = (course_term['course_fill_rate'] >= HIGH_FILL_THRESHOLD).astype(float)

# merge term index for chronological ordering
course_term = course_term.merge(
    terms[['ml_term_id', 'term_idx', 'term_order', 'year', 'term', 'is_covid_affected']],
    on='ml_term_id'
)

print(f'unique (course, term) pairs: {len(course_term):,}')
print(f'unique courses:              {course_term["ml_course_id"].nunique():,}')
print(f'unique terms:                {course_term["ml_term_id"].nunique()}')
print()
print('Sample:')
print(course_term.head(5))

unique (course, term) pairs: 24,376
unique courses:              3,067
unique terms:                19

Sample:
   ml_course_id  ml_term_id  n_sections  total_capacity  total_enrolled  \
0             1           3           1             120              77   
1             1           6           1             120              97   
2             1           9           1             120              54   
3             1          12           1             100              55   
4             1          13           1             120              53   

   avg_fill_rate  course_fill_rate  is_high_fill_term  term_idx  term_order  \
0       0.641667          0.641667                0.0         2           3   
1       0.808333          0.808333                0.0         5           3   
2       0.450000          0.450000                0.0         8           3   
3       0.550000          0.550000                0.0        11           3   
4       0.441667          0.441667        

---
## Block 3 — Build full grid (including negatives)

The offered model needs rows for courses that were NOT offered in a given term.  
For each course, we generate one row per term from its **first offering onward**.  
Terms before a course's first appearance are excluded — a course can't 'not be offered' before it exists.

In [4]:
# first term each course was ever offered
first_term_per_course = (
    course_term.groupby('ml_course_id')['term_idx']
    .min()
    .reset_index()
    .rename(columns={'term_idx': 'first_term_idx'})
)

all_term_idxs = sorted(terms['term_idx'].unique())

# build grid: for every course, every term from first_offering onward
grid_rows = []
for _, row in first_term_per_course.iterrows():
    cid       = int(row['ml_course_id'])
    first_idx = int(row['first_term_idx'])
    for tidx in all_term_idxs:
        if tidx >= first_idx:
            grid_rows.append({'ml_course_id': cid, 'term_idx': tidx})

grid = pd.DataFrame(grid_rows)
print(f'full grid: {len(grid):,} rows (course × term combos)')

# merge actual offerings — offered=1 rows get real data, offered=0 get NaN
grid = grid.merge(
    course_term[['ml_course_id', 'term_idx', 'n_sections', 'total_capacity',
                 'total_enrolled', 'avg_fill_rate', 'course_fill_rate', 'is_high_fill_term']],
    on=['ml_course_id', 'term_idx'],
    how='left'
)
grid['offered']    = grid['n_sections'].notna().astype(int)
grid['n_sections'] = grid['n_sections'].fillna(0).astype(int)

# merge term metadata (all rows need term info)
grid = grid.merge(
    terms[['term_idx', 'ml_term_id', 'term_order', 'year', 'term', 'is_covid_affected']],
    on='term_idx',
    how='left'
)

n_pos = grid['offered'].sum()
n_neg = (grid['offered'] == 0).sum()
print(f'offered=1: {n_pos:,}   offered=0: {n_neg:,}')
print(f'positive rate: {n_pos/len(grid)*100:.1f}%')

full grid: 48,907 rows (course × term combos)
offered=1: 24,376   offered=0: 24,531
positive rate: 49.8%


---
## Block 4 — Compute historical features

For each (course, term) row, we look **only at prior terms** (term_idx < current).  
This is the no-leakage guarantee — the model never sees future data during training.

In [5]:
ALL_HIST_FEATURES = [
    # offered model features
    'hist_n_offerings',
    'hist_n_this_semester_offerings',
    'same_semester_offer_ratio',
    'n_distinct_semesters_offered',
    'n_terms_since_last_offered',
    'n_consecutive_same_semester_streak',
    # capacity model features
    'hist_avg_capacity_per_offering',
    'hist_avg_capacity_this_semester',
    'same_semester_capacity_ratio',
    'previous_term_capacity',
    'previous_same_semester_capacity',
    'capacity_trend',
    'hist_avg_sections_per_offering',
    'hist_avg_enrollment_per_offering',
    # enrollment model features
    'hist_avg_enrollment_this_semester',
    'same_semester_enrollment_ratio',
    'previous_same_semester_enrollment',
    'previous_term_enrollment',
    'enrollment_trend',
    'high_fill_rate_frequency',
    'course_age_terms',
]

print(f'features to compute: {len(ALL_HIST_FEATURES)}')

features to compute: 21


In [6]:
def compute_course_features(group):
    """
    For a single course (all its rows across all terms), compute all historical features.
    Called via groupby('ml_course_id').apply().
    All features are computed using ONLY rows prior to the current one (no leakage).
    """
    group = group.sort_values('term_idx').reset_index(drop=True)
    n     = len(group)
    feats = {col: [np.nan] * n for col in ALL_HIST_FEATURES}

    for i in range(n):
        row          = group.iloc[i]
        prior        = group.iloc[:i]                                       # all prior rows
        prior_off    = prior[prior['offered'] == 1]                         # prior offered rows only
        same_sem     = prior[prior['term_order'] == row['term_order']]      # prior same-semester rows
        same_sem_off = prior_off[prior_off['term_order'] == row['term_order']]  # prior same-sem + offered

        n_prior_off  = len(prior_off)
        n_same_sem   = len(same_sem)
        n_same_off   = len(same_sem_off)

        # ── OFFERED features ──────────────────────────────────────────────

        feats['hist_n_offerings'][i]             = n_prior_off
        feats['hist_n_this_semester_offerings'][i] = n_same_off

        # out of all prior times this semester occurred, how often was it offered?
        feats['same_semester_offer_ratio'][i] = (
            n_same_off / n_same_sem if n_same_sem > 0 else np.nan
        )

        feats['n_distinct_semesters_offered'][i] = (
            int(prior_off['term_order'].nunique()) if n_prior_off > 0 else 0
        )

        # terms elapsed since last offered (NaN if never offered before)
        feats['n_terms_since_last_offered'][i] = (
            int(row['term_idx'] - prior_off['term_idx'].max()) if n_prior_off > 0 else np.nan
        )

        # consecutive same-semester streak — walk backwards through same-semester prior rows
        streak = 0
        for j in range(len(same_sem) - 1, -1, -1):
            if same_sem.iloc[j]['offered'] == 1:
                streak += 1
            else:
                break
        feats['n_consecutive_same_semester_streak'][i] = streak

        # ── CAPACITY / ENROLLMENT features ────────────────────────────────

        # how many terms old is this course in our dataset?
        feats['course_age_terms'][i] = i + 1

        if n_prior_off > 0:
            feats['hist_avg_capacity_per_offering'][i]  = prior_off['total_capacity'].mean()
            feats['hist_avg_enrollment_per_offering'][i] = prior_off['total_enrolled'].mean()
            feats['hist_avg_sections_per_offering'][i]  = prior_off['n_sections'].mean()
            feats['high_fill_rate_frequency'][i]        = (prior_off['is_high_fill_term'] == 1).mean()
            feats['previous_term_capacity'][i]           = prior_off.iloc[-1]['total_capacity']
            feats['previous_term_enrollment'][i]         = prior_off.iloc[-1]['total_enrolled']

            # linear trend: slope of capacity/enrollment over all prior offered terms
            # positive = growing, negative = shrinking
            if n_prior_off >= 2:
                x = np.arange(n_prior_off)
                feats['capacity_trend'][i]   = float(np.polyfit(x, prior_off['total_capacity'].values, 1)[0])
                feats['enrollment_trend'][i] = float(np.polyfit(x, prior_off['total_enrolled'].values, 1)[0])

        if n_same_off > 0:
            feats['hist_avg_capacity_this_semester'][i]   = same_sem_off['total_capacity'].mean()
            feats['hist_avg_enrollment_this_semester'][i] = same_sem_off['total_enrolled'].mean()
            feats['previous_same_semester_capacity'][i]   = same_sem_off.iloc[-1]['total_capacity']
            feats['previous_same_semester_enrollment'][i] = same_sem_off.iloc[-1]['total_enrolled']

        # ratios — only compute when denominator is non-zero
        cap_avg = feats['hist_avg_capacity_per_offering'][i]
        cap_sem = feats['hist_avg_capacity_this_semester'][i]
        if not np.isnan(cap_avg) and cap_avg > 0 and not np.isnan(cap_sem):
            feats['same_semester_capacity_ratio'][i] = cap_sem / cap_avg

        enr_avg = feats['hist_avg_enrollment_per_offering'][i]
        enr_sem = feats['hist_avg_enrollment_this_semester'][i]
        if not np.isnan(enr_avg) and enr_avg > 0 and not np.isnan(enr_sem):
            feats['same_semester_enrollment_ratio'][i] = enr_sem / enr_avg

    for col, vals in feats.items():
        group[col] = vals

    return group

print('Function defined. Run next cell to compute (takes ~1-3 min).')

Function defined. Run next cell to compute (takes ~1-3 min).


In [7]:
import time
t0 = time.time()

features = grid.groupby('ml_course_id', group_keys=False).apply(compute_course_features)
features = features.reset_index(drop=True)

print(f'done in {time.time()-t0:.1f}s')
print(f'shape: {features.shape}')
print(f'offered=1: {features["offered"].sum():,}   offered=0: {(features["offered"]==0).sum():,}')

done in 169.6s
shape: (48907, 35)
offered=1: 24,376   offered=0: 24,531


C:\Users\chait\AppData\Local\Temp\ipykernel_44844\2492516541.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features = grid.groupby('ml_course_id', group_keys=False).apply(compute_course_features)


---
## Block 5 — Verify: no leakage spot check

Pick a known course and confirm historical features are correct.

---
## Block 6 — Join static course features

In [10]:
features = features.merge(courses_meta, on='ml_course_id', how='left')

# verify no courses lost
null_dept = features['dept_code'].isna().sum()
print(f'rows with null dept_code after join: {null_dept}')
print(f'shape after join: {features.shape}')

rows with null dept_code after join: 0
shape after join: (48907, 40)


---
## Block 7 — Encode categoricals

In [11]:
# label encode dept_code and degree_level
# fit on ALL data — consistent mapping across train/test/final
# label encoding is safe here: tree models don't assume ordinality

le_dept   = LabelEncoder()
le_degree = LabelEncoder()

features['dept_code']    = features['dept_code'].fillna('UNKNOWN')
features['degree_level'] = features['degree_level'].fillna('UNKNOWN')

features['dept_code_enc']    = le_dept.fit_transform(features['dept_code'])
features['degree_level_enc'] = le_degree.fit_transform(features['degree_level'])

print(f'dept_code unique values:    {len(le_dept.classes_)}')
print(f'degree_level unique values: {len(le_degree.classes_)}')
print(f'degree_level classes: {le_degree.classes_}')

# save encoders for use in prediction pipeline
import joblib
joblib.dump(le_dept,   OUT_DIR / 'le_dept_code.pkl')
joblib.dump(le_degree, OUT_DIR / 'le_degree_level.pkl')
print('Encoders saved.')

dept_code unique values:    72
degree_level unique values: 2
degree_level classes: ['GRAD' 'UGRD']
Encoders saved.


---
## Block 8 — Cold-start handling

Courses appearing for the first time have no prior history → all `hist_*` features are NaN.  
Strategy: fill with **department-level averages** computed from all offered rows.  
This is better than dropping these rows — the model still learns from them with imputed context.

`n_terms_since_last_offered` when never offered: fill with total terms in dataset (signals 'very stale').

In [12]:
MAX_TERMS = len(all_term_idxs)  # 19 — used as 'never offered' sentinel

# compute dept-level averages from all offered rows
offered_rows = features[features['offered'] == 1]
dept_avgs = offered_rows.groupby('dept_code_enc')[[
    'hist_avg_capacity_per_offering',
    'hist_avg_capacity_this_semester',
    'hist_avg_enrollment_per_offering',
    'hist_avg_enrollment_this_semester',
    'hist_avg_sections_per_offering',
    'previous_term_capacity',
    'previous_same_semester_capacity',
    'previous_term_enrollment',
    'previous_same_semester_enrollment',
    'high_fill_rate_frequency',
]].mean()

# global averages as fallback (for departments with no history either)
global_avgs = offered_rows[dept_avgs.columns].mean()

print('Dept-level averages (sample):')
print(dept_avgs.head(3).round(1))

Dept-level averages (sample):
               hist_avg_capacity_per_offering  \
dept_code_enc                                   
0                                        23.0   
1                                        34.3   
2                                        19.7   

               hist_avg_capacity_this_semester  \
dept_code_enc                                    
0                                         23.4   
1                                         34.7   
2                                         19.7   

               hist_avg_enrollment_per_offering  \
dept_code_enc                                     
0                                          10.2   
1                                          27.2   
2                                           7.7   

               hist_avg_enrollment_this_semester  \
dept_code_enc                                      
0                                           10.1   
1                                           28.1   
2        

In [13]:
# fill NaNs in historical features
for col in dept_avgs.columns:
    null_mask = features[col].isna()
    if null_mask.sum() == 0:
        continue
    # map dept_code_enc → dept average
    fill_vals = features.loc[null_mask, 'dept_code_enc'].map(dept_avgs[col])
    # where dept avg is also NaN (new dept), fall back to global avg
    fill_vals = fill_vals.fillna(global_avgs[col])
    features.loc[null_mask, col] = fill_vals
    print(f'  {col}: filled {null_mask.sum():,} NaNs')

# fill ratio features (NaN when no same-semester history) with 0
for col in ['same_semester_offer_ratio', 'same_semester_capacity_ratio', 'same_semester_enrollment_ratio']:
    n = features[col].isna().sum()
    features[col] = features[col].fillna(0)
    print(f'  {col}: filled {n:,} NaNs with 0')

# fill trend features with 0 (no trend when < 2 prior offerings)
for col in ['capacity_trend', 'enrollment_trend']:
    n = features[col].isna().sum()
    features[col] = features[col].fillna(0)
    print(f'  {col}: filled {n:,} NaNs with 0')

# n_terms_since_last_offered: NaN means never offered → fill with MAX_TERMS
n = features['n_terms_since_last_offered'].isna().sum()
features['n_terms_since_last_offered'] = features['n_terms_since_last_offered'].fillna(MAX_TERMS)
print(f'  n_terms_since_last_offered: filled {n:,} NaNs with {MAX_TERMS}')

  hist_avg_capacity_per_offering: filled 3,067 NaNs
  hist_avg_capacity_this_semester: filled 22,742 NaNs
  hist_avg_enrollment_per_offering: filled 3,067 NaNs
  hist_avg_enrollment_this_semester: filled 22,742 NaNs
  hist_avg_sections_per_offering: filled 3,067 NaNs
  previous_term_capacity: filled 3,067 NaNs
  previous_same_semester_capacity: filled 22,742 NaNs
  previous_term_enrollment: filled 3,067 NaNs
  previous_same_semester_enrollment: filled 22,742 NaNs
  high_fill_rate_frequency: filled 3,067 NaNs
  same_semester_offer_ratio: filled 9,056 NaNs with 0
  same_semester_capacity_ratio: filled 22,742 NaNs with 0
  same_semester_enrollment_ratio: filled 22,742 NaNs with 0
  capacity_trend: filled 12,026 NaNs with 0
  enrollment_trend: filled 12,026 NaNs with 0
  n_terms_since_last_offered: filled 3,067 NaNs with 19


In [14]:
# verify no remaining NaNs in feature columns
feat_cols_all = ALL_HIST_FEATURES + ['dept_code_enc', 'degree_level_enc', 'course_level', 'units', 'prereq_count', 'term_order', 'is_covid_affected']
null_check = features[feat_cols_all].isnull().sum()
remaining  = null_check[null_check > 0]

if len(remaining) == 0:
    print('✓ No NaNs remaining in feature columns')
else:
    print('⚠ NaNs remaining:')
    print(remaining)

✓ No NaNs remaining in feature columns


---
## Block 9 — Define feature sets and targets

In [15]:
# note: dept_code_enc and degree_level_enc replace the raw string columns

FEATURES_OFFERED = [
    'ml_course_id',
    'dept_code_enc',
    'degree_level_enc',
    'course_level',
    'units',
    'term_order',
    'is_covid_affected',
    'hist_n_offerings',
    'hist_n_this_semester_offerings',
    'same_semester_offer_ratio',
    'n_distinct_semesters_offered',
    'n_terms_since_last_offered',
    'n_consecutive_same_semester_streak',
]

FEATURES_CAPACITY = [
    'ml_course_id',
    'dept_code_enc',
    'degree_level_enc',
    'course_level',
    'units',
    'term_order',
    'is_covid_affected',
    'hist_avg_capacity_per_offering',
    'hist_avg_capacity_this_semester',
    'same_semester_capacity_ratio',
    'previous_term_capacity',
    'previous_same_semester_capacity',
    'capacity_trend',
    'hist_n_offerings',
    'hist_avg_sections_per_offering',
    'hist_avg_enrollment_per_offering',
]

FEATURES_ENROLLMENT = [
    'ml_course_id',
    'dept_code_enc',
    'degree_level_enc',
    'course_level',
    'units',
    'term_order',
    'is_covid_affected',
    'hist_avg_capacity_per_offering',
    'hist_avg_capacity_this_semester',
    'hist_avg_enrollment_per_offering',
    'hist_avg_enrollment_this_semester',
    'same_semester_enrollment_ratio',
    'previous_same_semester_enrollment',
    'previous_term_enrollment',
    'enrollment_trend',
    'hist_n_offerings',
    'hist_avg_sections_per_offering',
    'high_fill_rate_frequency',
    'prereq_count',
    'course_age_terms',
]

print(f'OFFERED features:    {len(FEATURES_OFFERED)}')
print(f'CAPACITY features:   {len(FEATURES_CAPACITY)}')
print(f'ENROLLMENT features: {len(FEATURES_ENROLLMENT)}')

OFFERED features:    13
CAPACITY features:   16
ENROLLMENT features: 20


---
## Block 10 — Split and save CSVs

In [16]:
# add log-transformed targets for regression models
# log1p handles any edge case zeros gracefully
features['log_capacity'] = np.log1p(features['total_capacity'])
features['log_enrolled'] = np.log1p(features['total_enrolled'])

# ── OFFERED ───────────────────────────────────────────────────────────────────
# all rows (positives + negatives), target = offered (0 or 1)
offered_df = features[FEATURES_OFFERED + ['year', 'offered']].copy()

# ── CAPACITY ──────────────────────────────────────────────────────────────────
# only offered=1 rows (can't predict capacity for unoffered courses)
# target = log_capacity (inverse: np.expm1 to get back real seats)
capacity_df = features[features['offered'] == 1][FEATURES_CAPACITY + ['year', 'total_capacity', 'log_capacity']].copy()

# ── ENROLLMENT ────────────────────────────────────────────────────────────────
enrollment_df = features[features['offered'] == 1][FEATURES_ENROLLMENT + ['year', 'total_enrolled', 'log_enrolled']].copy()

print('Dataset sizes before split:')
print(f'  offered:    {len(offered_df):,}')
print(f'  capacity:   {len(capacity_df):,}')
print(f'  enrollment: {len(enrollment_df):,}')

Dataset sizes before split:
  offered:    48,907
  capacity:   24,376
  enrollment: 24,376


In [17]:
def split_and_save(df, name, train_cutoff, test_year, final_year, out_dir):
    train = df[df['year'] <= train_cutoff].drop(columns='year')
    test  = df[df['year'] == test_year].drop(columns='year')
    final = df[df['year'] == final_year].drop(columns='year')

    train.to_csv(out_dir / f'{name}_train.csv', index=False)
    test.to_csv( out_dir / f'{name}_test.csv',  index=False)
    final.to_csv(out_dir / f'{name}_final.csv', index=False)

    print(f'{name:12s}: train={len(train):>5,}  test={len(test):>4,}  final={len(final):>4,}')

print('Saving CSVs...')
split_and_save(offered_df,    'offered',    TRAIN_CUTOFF, TEST_YEAR, FINAL_YEAR, OUT_DIR)
split_and_save(capacity_df,   'capacity',   TRAIN_CUTOFF, TEST_YEAR, FINAL_YEAR, OUT_DIR)
split_and_save(enrollment_df, 'enrollment', TRAIN_CUTOFF, TEST_YEAR, FINAL_YEAR, OUT_DIR)
print()
print('Files saved to:', OUT_DIR)

Saving CSVs...
offered     : train=36,897  test=8,943  final=3,067
capacity    : train=18,728  test=4,126  final=1,522
enrollment  : train=18,728  test=4,126  final=1,522

Files saved to: ..\data\processed


In [18]:
# final check — list all output files
for f in sorted(OUT_DIR.glob('*.csv')):
    size_kb = f.stat().st_size // 1024
    print(f'{f.name:<35}  {size_kb:>5} KB')

capacity_final.csv                     202 KB
capacity_test.csv                      514 KB
capacity_train.csv                    2354 KB
enrollment_final.csv                   241 KB
enrollment_test.csv                    596 KB
enrollment_train.csv                  2781 KB
offered_final.csv                      130 KB
offered_test.csv                       360 KB
offered_train.csv                     1492 KB


In [19]:
# sanity check: read back one CSV and verify it looks correct
sample = pd.read_csv(OUT_DIR / 'offered_train.csv')
print('offered_train.csv:')
print(f'  shape: {sample.shape}')
print(f'  offered=1: {sample["offered"].sum():,}  offered=0: {(sample["offered"]==0).sum():,}')
print(f'  null count: {sample.isnull().sum().sum()}')
print()
print(sample.head(3))

offered_train.csv:
  shape: (36897, 14)
  offered=1: 18,728  offered=0: 18,169
  null count: 0

   ml_course_id  dept_code_enc  degree_level_enc  course_level  units  \
0             1              0                 1           100    3.0   
1             1              0                 1           100    3.0   
2             1              0                 1           100    3.0   

   term_order  is_covid_affected  hist_n_offerings  \
0           3                  1                 0   
1           1                  1                 1   
2           2                  1                 1   

   hist_n_this_semester_offerings  same_semester_offer_ratio  \
0                               0                        0.0   
1                               0                        0.0   
2                               0                        0.0   

   n_distinct_semesters_offered  n_terms_since_last_offered  \
0                             0                        19.0   
1          